In [ ]:
# The pre-requisite for this python file is ncat to be installed 
#docker exec -it <jupyter-lab-process-id>
#sudo apt-get install ncat

In [1]:
from pyspark.sql import SparkSession

spark=SparkSession\
.builder \
.appName("read from socket") \
.master("local[*]") \
.getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions",8)

In [2]:
#Read Input
df_raw=spark.readStream.format("socket").option("host","localhost").option("port","9999").load()
# df_raw.printSchema()

In [4]:
#
from pyspark.sql.functions import split

df_splt=df_raw.withColumn("words",split("value"," "))
# df_splt.show()

In [5]:
#explode
from pyspark.sql.functions import explode

df_explode=df_splt.withColumn("word",explode("words")).drop("value","words")
# df_explode.show()

In [6]:
from pyspark.sql.functions import lit,count
df_agg=df_explode.groupBy("word").agg(count(lit (1)).alias("CNT"))
# df_agg.show()

In [7]:
#WriteStream-update
df_agg.writeStream.format("console").outputMode("update").start().awaitTermination

<bound method StreamingQuery.awaitTermination of <pyspark.sql.streaming.StreamingQuery object at 0x779ae8ccbb50>>